# Bridge 03 — Coordinate Math, Greedy Patterns, Spatial Ops

Mechanics that block dojo 06 (conv loop, K-means) and dojo 07 (IoU, NMS, rotation, voxels).  
No hints. If stuck > 10 min: write the loop version first, then vectorize.

In [ ]:
import numpy as np
np.random.seed(42)

---
## Section A — Box Coordinate Arithmetic

### A1 — box area
Given `boxes` of shape `(N, 4)` in format `[x1, y1, x2, y2]`, compute area of each box — shape `(N,)`.

In [ ]:
boxes = np.array([[0,0,4,3],[1,1,5,4],[10,10,12,13]], dtype=float)

areas = None  # YOUR CODE

assert areas.shape == (3,)
assert np.allclose(areas, [12, 12, 6]), areas

### A2 — intersection of two boxes
Given two boxes `a` and `b` each as `[x1, y1, x2, y2]`, compute the area of their intersection. Returns 0 if they don't overlap.

In [ ]:
def intersection_area(a, b):
    """a, b: 1D arrays [x1, y1, x2, y2]. Returns scalar."""
    # Intersection box: top-left = max of top-lefts, bottom-right = min of bottom-rights
    # YOUR CODE
    pass

assert intersection_area([0,0,4,4], [2,2,6,6]) == 4.0    # 2x2 overlap
assert intersection_area([0,0,4,4], [5,5,8,8]) == 0.0    # no overlap
assert intersection_area([0,0,4,4], [1,1,3,3]) == 4.0    # fully contained

### A3 — single-pair IoU
IoU = intersection / union = intersection / (area_a + area_b - intersection).

In [ ]:
def iou(a, b):
    """a, b: [x1,y1,x2,y2]. Returns scalar in [0,1]."""
    # YOUR CODE
    pass

assert np.isclose(iou([0,0,4,4], [0,0,4,4]), 1.0)  # identical
assert np.isclose(iou([0,0,4,4], [5,0,9,4]), 0.0)  # no overlap
# 50% horizontal overlap on same-size boxes
val = iou([0,0,4,4], [2,0,6,4])
assert np.isclose(val, 8/24, atol=1e-4), val  # intersection=8, union=24

### A4 — vectorized intersection: `(N, 1, 4)` vs `(1, M, 4)`
Given `boxes_a` of shape `(N, 4)` and `boxes_b` of shape `(M, 4)`, compute the intersection area for all `N×M` pairs — shape `(N, M)`. No loops.

In [ ]:
boxes_a = np.array([[0,0,4,4],[2,2,6,6]], dtype=float)   # (2, 4)
boxes_b = np.array([[1,1,3,3],[0,0,4,4],[5,5,9,9]], dtype=float)  # (3, 4)

# YOUR CODE — shape (2, 3)
inter = None

assert inter.shape == (2, 3), inter.shape
# Verify against single-pair function
for i in range(2):
    for j in range(3):
        expected = intersection_area(boxes_a[i], boxes_b[j])
        assert np.isclose(inter[i, j], expected), f'[{i},{j}]: {inter[i,j]} vs {expected}'

### A5 — full vectorized IoU matrix
Build on A4 to compute the full `(N, M)` IoU matrix.

In [ ]:
def iou_matrix(a, b):
    """a: (N,4), b: (M,4) → (N,M)"""
    # YOUR CODE
    pass

iou_mat = iou_matrix(boxes_a, boxes_b)

assert iou_mat.shape == (2, 3)
assert iou_mat.min() >= 0 and iou_mat.max() <= 1
# Self-IoU: a box with itself = 1
self_iou = iou_matrix(boxes_a, boxes_a)
assert np.allclose(np.diag(self_iou), 1.0)

---
## Section B — Greedy Loop Patterns

### B1 — greedy set cover
Given scores and a conflict matrix `C[i,j]=1` meaning i and j conflict (can't both be selected), greedily pick items by descending score, skipping items that conflict with already-selected ones. Return list of kept indices.

In [ ]:
scores   = np.array([0.9, 0.8, 0.7, 0.6, 0.5])
conflict = np.array([
    [0,1,0,0,0],
    [1,0,1,0,0],
    [0,1,0,0,1],
    [0,0,0,0,1],
    [0,0,1,1,0],
])

def greedy_select(scores, conflict):
    """
    Returns list of kept indices.
    Algorithm:
      1. Sort by score descending
      2. Pick the top-score remaining item
      3. Remove all items that conflict with it
      4. Repeat
    """
    # YOUR CODE
    pass

kept = greedy_select(scores, conflict)

assert set(kept) == {0, 3}, f'Expected {{0, 3}}, got {kept}'
# Verify: no two kept items conflict
for i in kept:
    for j in kept:
        if i != j:
            assert conflict[i, j] == 0, f'Items {i} and {j} conflict'

### B2 — K-means one iteration
Given `X` of shape `(N, D)` and current `centroids` of shape `(K, D)`:  
1. Assign each point to nearest centroid (using pairwise L2) — shape `(N,)`
2. Update centroids as mean of assigned points — shape `(K, D)`
3. Handle empty clusters by keeping the old centroid

In [ ]:
X = np.array([[0,0],[1,0],[0,1],[5,5],[6,5],[5,6]], dtype=float)
centroids = np.array([[0.5, 0.5],[5.5, 5.5]], dtype=float)

def kmeans_step(X, centroids):
    """
    Returns:
        assignments: (N,) int
        new_centroids: (K, D)
    """
    K = len(centroids)
    # Step 1: YOUR CODE — pairwise L2, then argmin
    
    # Step 2: YOUR CODE — mean per cluster
    # Handle empty: if no points assigned to cluster k, keep centroids[k]
    
    pass

assignments, new_centroids = kmeans_step(X, centroids)

assert assignments.shape == (6,)
assert np.allclose(assignments, [0, 0, 0, 1, 1, 1]), assignments
assert np.allclose(new_centroids[0], [1/3, 1/3], atol=1e-4), new_centroids[0]
assert np.allclose(new_centroids[1], [16/3, 16/3], atol=1e-4), new_centroids[1]

### B3 — convergence check
Given old and new assignment arrays, return `True` if they are identical (convergence criterion for K-means).

In [ ]:
old = np.array([0, 0, 1, 1, 2])
new_same = np.array([0, 0, 1, 1, 2])
new_diff = np.array([0, 1, 1, 1, 2])

def converged(old, new):
    # YOUR CODE — one expression
    pass

assert converged(old, new_same) == True
assert converged(old, new_diff) == False

---
## Section C — Rotation & Homogeneous Coordinates

### C1 — 2D rotation matrix
Build the 2D rotation matrix for angle `theta` (CCW, radians) → `(2, 2)`.

In [ ]:
def R2(theta):
    # YOUR CODE
    pass

# 90° CCW: (1,0) → (0,1)
assert np.allclose(R2(np.pi/2) @ [1,0], [0,1], atol=1e-6)
# 180°: (1,0) → (-1,0)
assert np.allclose(R2(np.pi) @ [1,0], [-1,0], atol=1e-6)
# Orthogonality
assert np.allclose(R2(0.5) @ R2(0.5).T, np.eye(2), atol=1e-6)
assert np.isclose(np.linalg.det(R2(0.5)), 1.0)

### C2 — rotate a batch of 2D points
Given `points` of shape `(N, 2)` and angle `theta`, rotate all points — shape `(N, 2)`. No loops.

In [ ]:
points = np.array([[1,0],[0,1],[-1,0],[0,-1]], dtype=float)
theta  = np.pi / 2  # 90° CCW

rotated = None  # YOUR CODE — shape (4, 2)

assert rotated.shape == (4, 2)
assert np.allclose(rotated[0], [0, 1],  atol=1e-6)  # (1,0)  → (0,1)
assert np.allclose(rotated[1], [-1, 0], atol=1e-6)  # (0,1)  → (-1,0)
assert np.allclose(rotated[2], [0, -1], atol=1e-6)  # (-1,0) → (0,-1)

### C3 — 3D rotation around Z-axis
Rz(theta) rotates in the XY-plane, Z unchanged: `[[c,-s,0],[s,c,0],[0,0,1]]`.

In [ ]:
def Rz(theta):
    # YOUR CODE — (3, 3)
    pass

# Rotating (1,0,0) by 90° CCW around Z gives (0,1,0)
assert np.allclose(Rz(np.pi/2) @ [1,0,0], [0,1,0], atol=1e-6)
# Z component unchanged
assert np.allclose(Rz(1.2) @ [0,0,5], [0,0,5], atol=1e-6)
# Orthogonality
R = Rz(0.7)
assert np.allclose(R @ R.T, np.eye(3), atol=1e-6)
assert np.isclose(np.linalg.det(R), 1.0)

### C4 — homogeneous coordinates
Convert `points` of shape `(N, 3)` to homogeneous `(N, 4)` by appending a column of ones. Then apply a 4×4 transform matrix and convert back to `(N, 3)`.

In [ ]:
points = np.array([[1,0,0],[0,1,0],[0,0,1]], dtype=float)

# Translation matrix: translate by (1, 2, 3)
T = np.eye(4)
T[:3, 3] = [1, 2, 3]

# Step 1: to homogeneous — append ones column
pts_h = None  # YOUR CODE — shape (3, 4)

# Step 2: apply T
pts_transformed_h = None  # YOUR CODE — shape (3, 4)

# Step 3: back to 3D — drop last column
pts_out = None  # YOUR CODE — shape (3, 3)

assert pts_h.shape == (3, 4)
assert np.allclose(pts_h[:, 3], 1.0)  # homogeneous coord = 1
assert pts_out.shape == (3, 3)
# Translation adds (1,2,3) to each point
assert np.allclose(pts_out, points + np.array([1,2,3])), pts_out

---
## Section D — Voxelization & Grid Ops

### D1 — bin 1D values into cells
Given continuous values `x`, assign each to a cell index using `floor(x / cell_size)`. Clip to `[0, num_cells-1]`.

In [ ]:
x = np.array([-0.5, 0.0, 0.99, 1.0, 2.5, 3.9, 4.0])
cell_size = 1.0
num_cells = 4  # valid range [0,3]

cell_idx = None  # YOUR CODE — shape (7,), clipped integers

assert cell_idx.dtype in [np.int32, np.int64, int]
assert np.allclose(cell_idx, [0, 0, 0, 1, 2, 3, 3]), cell_idx  # clip negatives to 0, >3 to 3

### D2 — 2D voxelization
Given 2D points `(N, 2)`, assign each to a grid cell `(i, j)` and count points per cell — return a `(H, W)` count array.

In [ ]:
points_2d = np.array([[0.5,0.5],[0.5,1.5],[1.5,0.5],[1.5,1.5],[1.5,1.5],[0.5,0.5]])
H, W = 2, 2
cell_size = 1.0

# Step 1: compute (row_idx, col_idx) per point
# Step 2: compute flat index = row * W + col
# Step 3: use np.bincount to count, reshape to (H, W)
grid = None  # YOUR CODE — shape (2, 2)

assert grid.shape == (2, 2)
assert grid.sum() == 6
assert np.allclose(grid, [[2, 1], [1, 2]]), grid  # top-left=2, top-right=1, ...

### D3 — max reduction per cell
Same points but instead of count, compute the **maximum** `z` value per cell. Use `np.maximum.at`.

In [ ]:
# Same (row, col) -> flat index from D2
z_vals = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])  # z per point
# flat indices from D2: points map to cells [0, 1, 2, 3, 3, 0]
flat_idx = np.array([0, 1, 2, 3, 3, 0])

max_z = np.zeros(4)  # 4 cells
# YOUR CODE — use np.maximum.at

assert np.allclose(max_z, [6., 2., 3., 5.]), max_z

---
## Section E — Sliding Window (Conv Mechanics)

### E1 — 1D sliding window sum
Given `x` of shape `(N,)` and window `w`, compute the sum of each window of size `w` — output shape `(N - w + 1,)`. Use only a loop over positions (understand the indexing before optimizing).

In [ ]:
x = np.array([1, 2, 3, 4, 5, 6], dtype=float)
w = 3

out = None  # YOUR CODE — shape (4,)

assert out.shape == (4,)
assert np.allclose(out, [6, 9, 12, 15]), out  # 1+2+3, 2+3+4, ...

### E2 — 2D sliding window: extract all patches
Given `img` of shape `(H, W)` and kernel size `(kH, kW)`, extract all `(H-kH+1) * (W-kW+1)` patches as a list/array. Each patch is shape `(kH, kW)`.

In [ ]:
img = np.arange(25, dtype=float).reshape(5, 5)
kH, kW = 3, 3

patches = []  # fill with all patches using nested loops
# YOUR CODE

assert len(patches) == (5-kH+1) * (5-kW+1)  # 9 patches
assert np.array(patches[0]).shape == (kH, kW)
# First patch: top-left 3×3
assert np.allclose(patches[0], img[:kH, :kW]), patches[0]

### E3 — single output pixel of 2D conv
Compute the convolution output at position `(i=1, j=1)` for `img` with `kernel`. This is `sum(img[i:i+kH, j:j+kW] * kernel)`.

In [ ]:
img    = np.arange(25, dtype=float).reshape(5, 5)
kernel = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=float)  # Laplacian
i, j   = 1, 1

out_pixel = None  # YOUR CODE — scalar

# Manual: patch at (1,1) is [[1,2,3],[6,7,8],[11,12,13]]
# Laplacian: 0*1 + 1*2 + 0*3 + 1*6 + (-4)*7 + 1*8 + 0*11 + 1*12 + 0*13
# = 2 + 6 - 28 + 8 + 12 = 0
assert np.isclose(out_pixel, 0.0), out_pixel

### E4 — full 2D conv (valid padding)
Put it together: compute the full output of shape `(H-kH+1, W-kW+1)` using nested loops over `(i, j)`.

In [ ]:
def conv2d(img, kernel):
    H, W = img.shape
    kH, kW = kernel.shape
    out_H, out_W = H - kH + 1, W - kW + 1
    out = np.zeros((out_H, out_W))
    # YOUR CODE — two nested loops + elementwise multiply + sum
    return out

# Test with identity kernel
identity = np.array([[0,0,0],[0,1,0],[0,0,0]], dtype=float)
result = conv2d(img, identity)
assert result.shape == (3, 3)
assert np.allclose(result, img[1:4, 1:4]), result  # identity conv = center crop

# Test with actual kernel
try:
    from scipy.signal import correlate2d
    ref = correlate2d(img, kernel, mode='valid')
    assert np.allclose(conv2d(img, kernel), ref, atol=1e-8), 'Mismatch vs scipy'
    print('E4 validated against scipy ✓')
except ImportError:
    print('scipy not available, skipping reference check')